In [24]:
import mysql.connector

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Piku@2026',
    port=3306
)
cursor = conn.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS retail_analysis")
cursor.execute("USE retail_analysis")
print("Database created!")

Database created!


In [25]:
cursor.execute("USE retail_analysis")
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    age INT,
    gender VARCHAR(10),
    item_purchased VARCHAR(50),
    category VARCHAR(30),
    purchase_amount DECIMAL(10,2),
    location VARCHAR(50),
    size VARCHAR(5),
    color VARCHAR(30),
    season VARCHAR(20),
    review_rating DECIMAL(3,1),
    subscription_status VARCHAR(5),
    shipping_type VARCHAR(30),
    discount_applied VARCHAR(5),
    promo_code_used VARCHAR(5),
    previous_purchases INT,
    payment_method VARCHAR(30),
    frequency_of_purchases VARCHAR(20)
)
""")
print("Table created!")

Table created!


In [29]:
import pandas as pd
import mysql.connector

# --- CONFIGURATION ---
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Piku@2026',
    'port': 3306,
    'database': 'retail_analysis'
}
TABLE_NAME = 'customers' 
FILE_PATH = '../data/cleaned_shopping_data.csv'

def import_data():
    conn = None
    cursor = None
    try:
        # 1. Establish Connection
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()

        # 2. Load and Clean Data
        df = pd.read_csv(FILE_PATH)
        # Replacing NaNs with None is correct for MySQL NULL values
        df = df.where(pd.notnull(df), None)

        # 3. Prepare SQL Query
        # Wrap column names in backticks (`) to handle names with spaces or reserved words
        cols = [f"`{col}`" for col in df.columns]
        columns_str = ", ".join(cols)
        placeholders = ", ".join(["%s"] * len(df.columns))
        sql = f"INSERT INTO {TABLE_NAME} ({columns_str}) VALUES ({placeholders})"

        print(f" Loading {len(df):,} rows into table '{TABLE_NAME}'...")

        # 4. Insert in batches using a transaction block
        batch_size = 1000
        total = len(df)
        
        for i in range(0, total, batch_size):
            batch = df.iloc[i:i+batch_size]
            rows = [tuple(row) for row in batch.to_numpy()]
            
            cursor.executemany(sql, rows)
            
            if (i + batch_size) % 10000 == 0 or (i + batch_size) >= total:
                print(f"   Progress: {min(i + batch_size, total):,} of {total:,} rows inserted...")

        # 5. Finalize
        conn.commit()
        print(f"\n SUCCESS: Data import complete!")

    except mysql.connector.Error as err:
        if conn: conn.rollback() # Rollback on error to keep DB clean
        print(f" Database Error: {err}")
    except Exception as e:
        print(f" Unexpected Error: {e}")
    finally:
        if cursor: cursor.close()
        if conn: conn.close()
        print(" Connection closed.")

if __name__ == "__main__":
    import_data()

 Loading 3,900 rows into table 'customers'...
   Progress: 3,900 of 3,900 rows inserted...

 SUCCESS: Data import complete!
 Connection closed.
